# Lab 7 — Grad-CAM, FGSM, PCA/SVD
AI Lab experiment notebook. Run cells in order.


In [ ]:

"""PCA and SVD dimensionality reduction."""
import numpy as np
from sklearn.datasets import load_iris
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

X=load_iris().data
X=StandardScaler().fit_transform(X)
pca=PCA(n_components=2)
X_pca=pca.fit_transform(X)
print("Original shape:",X.shape)
print("PCA shape:",X_pca.shape)
print("Explained variance ratio:",pca.explained_variance_ratio_)

U,S,Vt=np.linalg.svd(X,full_matrices=False)
X_svd=U[:,:2] @ np.diag(S[:2])
print("SVD reduced shape:",X_svd.shape)



"""FGSM adversarial example demonstration using a small synthetic PyTorch model."""
import torch

def fgsm_attack(image, epsilon, data_grad):
    perturbed=image + epsilon*data_grad.sign()
    return torch.clamp(perturbed,0,1)

if __name__=="__main__":
    # Demonstration input and gradient; replace with a real trained image classifier.
    image=torch.rand(1,3,32,32)
    image.requires_grad=True
    model=torch.nn.Sequential(torch.nn.Flatten(),torch.nn.Linear(3*32*32,10))
    target=torch.tensor([2])
    loss=torch.nn.CrossEntropyLoss()(model(image),target)
    model.zero_grad(); loss.backward()
    adversarial=fgsm_attack(image.detach(),0.03,image.grad)
    print("Input range:",image.min().item(),image.max().item())
    print("Adversarial range:",adversarial.min().item(),adversarial.max().item())
    print("Perturbation L-infinity:",(adversarial-image.detach()).abs().max().item())
